# Jev versus eleven conventional classification pipelines — v3

Use a **fresh Kaggle session**, Internet enabled, **T4 × 2**, and an enabled secret named `TYPESAFE_API_KEY`.

This notebook extracts its bundled Python modules; no separate code upload is required. The full editable bundle is also available. All ML tuning uses development data. One fixed test set is shared across models and training seeds. The previous run should be retained as a pilot.

**Run the ML cells first.** The separate Jev cell makes paid API requests. Full mode plans about 52,410 requests before cache reuse/retries on the current snapshots. Check the displayed counts and your pricing. The estimate guard is not a billing cap.

This is a modest-budget comparison, not a claim of best achievable ML performance.

## Fast preset — reduced CPU workloads
All 11 models and four candidates remain. Tree ceilings, text dimensions and training caps are reduced; accuracy may change. GPU models run first. Other sklearn models use CPUs. There is no measured 30-minute guarantee.
cuML accelerates compatible estimators; remaining cases have explicit CPU labels.
Read BENCHMARK_V3.md displayed below for the exact differences. Use a NEW output directory and preserve the old results.

In [1]:
# Extract the frozen source package. Run before importing numerical libraries.
import base64, hashlib, io, sys, zipfile
from pathlib import Path

# Optional: point at an edited bundle uploaded as a Kaggle Dataset.
# The directory must contain jevbench/, requirements-v2.txt, and BENCHMARK_V2.md.
EXTERNAL_CODE_DIR = None
PACKAGE_SHA256 = '518f37882398a19048be3811e75319246d138060b01155e53f76e0155a0ea2bf'
if EXTERNAL_CODE_DIR:
    CODE_DIR = Path(EXTERNAL_CODE_DIR)
else:
    bundled = base64.b64decode('...')
    assert hashlib.sha256(bundled).hexdigest() == PACKAGE_SHA256
    CODE_DIR = Path('/kaggle/working') / ('jevbench_code_' + PACKAGE_SHA256[:12])
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(bundled)) as archive:
        archive.extractall(CODE_DIR)
assert (CODE_DIR / 'jevbench' / 'runner.py').exists()
sys.path.insert(0, str(CODE_DIR))
print('Source modules:', CODE_DIR)

Source modules: /kaggle/working/jevbench_code_518f37882398


In [2]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_DIR / 'requirements-v2.txt')])
# If you imported numpy/sklearn before installation, restart the session and rerun.

# CUDA imports and actual GPU checks happen ONLY in isolated child processes.
import importlib.util
if importlib.util.find_spec('cuml') is None:
    raise RuntimeError('cuML missing: select Kaggle latest GPU environment.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires xgboost>=3.0.0, but you have xgboost 2.1.4 which is incompatible.


## Freeze the run configuration

`v3` uses three seeds, four candidates, up to 150 trees (60 histogram iterations), 8,000 training rows, 1,000 model-selection rows, and 500 decision-policy rows. `quick` is for smoke testing only. Test caps are 1,000 per dataset and 1,500 for Banking77; small datasets retain fewer rows.

Default pilot exclusion assumes seeds 42/43/44, test cap 300 and matching snapshots. If your pilot differed, use its actual saved IDs in `datasets.py` before running. Changing the configuration, source or environment requires a new `ROOT`.

In [3]:
from IPython.display import display, Markdown, FileLink
from jevbench.config import configuration
from jevbench.runner import prepare_suite, run_ml, run_jev
from jevbench.reporting import render_results

CFG = configuration('v3')
ROOT = Path('/kaggle/working/jev_benchmark_v3_1')
# Optional existing pilot output directory: reuses data snapshots, not pilot scores.
PILOT_ROOT = Path('/kaggle/working/jev_benchmark_v3')
if not PILOT_ROOT.exists():
    PILOT_ROOT = None
# Example: PILOT_ROOT = Path('/kaggle/input/my-pilot-results/jev_benchmark_parallel')

# Two T4s: independent worker per GPU; CPU models use two threads per worker.
CFG['max_parallel_jobs'] = 2
CFG['threads'] = 2
CFG['jev_workers'] = 8  # baked-in default
CFG['min_request_interval'] = 0.15
CFG['jev_model'] = 'jev-1.13.0'
display(CFG)
display(Markdown((CODE_DIR / 'BENCHMARK_V3.md').read_text()))

{'protocol': '3.0.1',
 'preset': 'v3',
 'datasets': ['AG News',
  'Banking77',
  'SMS Spam',
  'IMDb',
  'Bank Marketing',
  'Online Shoppers',
  'Breast Cancer',
  'Iris'],
 'seeds': [2027, 2028, 2029],
 'holdout_seed': 20260920,
 'train_cap': 8000,
 'validation_cap': 1000,
 'policy_cap': 500,
 'test_cap': 1000,
 'banking_test_cap': 1500,
 'exclude_pilot_tests': True,
 'pilot_seeds': [42, 43, 44],
 'pilot_test_cap': 300,
 'max_trials': 4,
 'trees': 150,
 'early_stopping': 15,
 'threads': 2,
 'max_parallel_jobs': 2,
 'word_features': 20000,
 'char_features': 10000,
 'selected_features': 512,
 'svd_components': 32,
 'max_text_chars': 4000,
 'threshold_quantiles': 101,
 'jev_model': 'jev-1.13.0',
 'jev_modes': ['zero-shot', 'few-shot'],
 'examples_per_class': 1,
 'jev_workers': 8,
 'min_request_interval': 0.15,
 'max_api_attempts': 65000,
 'max_retries': 2,
 'estimated_usd_per_million_input_tokens': 0.042,
 'max_estimated_api_usd': 20.0,
 'bootstrap_samples': 1000,
 'speed_profile': True

# Jev classification benchmark v3 (3.0.1 patch)

3.0.1 fixes device isolation. Native-library/cuML probes and training now run in fresh subprocesses with one `CUDA_VISIBLE_DEVICES` entry set before imports. The two physical GPUs each become local device 0 in their own process. Both probe subprocesses must succeed before any training subprocess starts; logs are saved under `gpu_workers/`. Interrupting the parent terminates its active children. The probe stage has a 180-second timeout. Hyperparameters, split selection and eight Jev workers are unchanged.

Upload `jev_benchmark_v3.ipynb`. It contains its Python modules. Use Internet on, T4 x2, **Kaggle Settings > Environment Preferences > Always use latest environment**, and your existing Kaggle secret. This uses Kaggle's preinstalled RAPIDS rather than blindly installing a CUDA wheel. The setup checks cuML imports; real adapter fits on both GPUs run before expensive training. Keep the old notebook/results as a separate experiment.

The original v2 increased the number of candidates from two to four and the tree ceiling from 200 to 400, while increasing training size and text tree features. CPU histogram boosting on Banking77 builds one tree per class per iteration: 400 iterations can mean 30,800 trees per candidate, before refitting. GPU idleness during these CPU models is expected.

The **v3 preset** retains all 11 models, three seeds, four candidates per family, class-weight comparisons, independent policy thresholds and the same test selection. Its declared computational budget is smaller:

| Setting | Original v2 | v3 |
|---|---:|---:|
| Training cap | 12,000 | 8,000 |
| Selection-validation cap | 1,500 | 1,000 |
| Forest/boosting tree ceiling | 400 | 150 |
| Histogram-boosting iteration ceiling | 400 | 60 |
| Histogram bins | 255 | 63 |
| Text histogram-boosting features | 2,000 selected TF-IDF | 32 scaled SVD components |
| Other text-tree features | 2,000 | 512 selected TF-IDF |
| Word/character vocabulary caps | 30,000 / 20,000 | 20,000 / 10,000 |
| SVD dimensions for k-NN | 64 | 32 |
| Forest second candidate depth/features | unlimited / 50% | 24 / 15% |
| Jev request workers | 4 | 8 |

These are runtime/representation tradeoffs, not mathematically equivalent accelerations; accuracy can change. The four-candidate search remains two parameter settings crossed with ordinary/balanced weights (k-NN uses four neighbor settings). See `BENCHMARK_V2.md` for the common protocol; **this document overrides its budget/representation/backend values for v3**.

## Explicit GPU backends

| Family | v3 backend |
|---|---|
| Logistic regression | cuML GPU, including sparse text; QN solver, same C candidates |
| Random forest | cuML GPU for unweighted candidates (128 bins, one stream); sklearn CPU for sample-weighted candidates |
| k-NN | cuML GPU brute-force neighbors, same distance/uniform weighting candidates |
| SVM | cuML GPU RBF on binary tabular tasks; sklearn for sparse text LinearSVC and multiclass tabular SVC |
| XGBoost / CatBoost | Their native CUDA implementations |
| Decision tree / Extra trees / Naive Bayes / Histogram gradient boost | sklearn CPU |
| Voting ensemble | CPU probability averaging of the three fitted members |

Backend routing is explicit rather than `cuml.accel` monkey-patching. No weighted random forest candidate silently loses its sample weights. GPU RF uses the dense form of the SAME selected TF-IDF values. cuML RF uses quantile-based splits and differs algorithmically from sklearn RF: this column selects among four **random-forest pipeline candidates with declared mixed implementations**. Do not describe it as pure sklearn or pure cuML. GPU logistic regression also uses a different solver. Each trial/result records its backend and estimator class; `run_diagnostics.csv` records the selected backend. Package/cuML/CuPy versions and GPU probe results are saved.

Each fresh worker sees only its assigned GPU; a CuPy context alone is no longer used to isolate devices. Failed cuML imports or preflight fits stop before the long benchmark. Sparse text LinearSVC, weighted forests and the other listed CPU cases remain CPU by explicit policy. Small datasets may be slower on GPU due to transfer/initialization overhead.

References: [RAPIDS on Kaggle](https://docs.nvidia.com/datascience/deployment/latest/platforms/kaggle/), [cuML compatibility restrictions](https://docs.nvidia.com/cuml/latest/cuml-accel/compatibility/), [GPU logistic regression](https://docs.nvidia.com/cuml/latest/api/generated/cuml.linear_model.LogisticRegression/), [GPU neighbors](https://docs.nvidia.com/cuml/latest/api/generated/cuml.neighbors.KNeighborsClassifier/).

Histogram boosting now enables sklearn's internal early stopping using 15% of the training partition. That subset remains within training, separate from model-selection validation, policy and test data. Model-selection validation still selects the candidate. The selected number of iterations is then fixed for the train+validation refit, with early stopping disabled. XGBoost/CatBoost use 15-round patience; their early stopping uses model-selection validation as before.

Both GPU libraries are probed before the run; the fast preset stops if no GPU passes instead of silently running boosting on CPU. XGBoost's actual fitted device is also checked. GPU estimators run first within each dataset/seed job. Each GPU has one worker; CPU models remain CPU models. This does not promise continuous GPU utilization or GPU acceleration of every algorithm. Per-candidate start/end times and CPU/GPU labels show where time is being spent. Failed trials remain visible.

## Switch from the interrupted run

1. Preserve/download the existing `/kaggle/working/jev_benchmark_v2` output folder before resetting a Kaggle session. Completed model results remain useful as old-protocol results.
2. Upload the v3 notebook. If the old folder remains accessible, set `PILOT_ROOT` in its configuration cell to that directory to reuse dataset snapshots. Otherwise leave it `None` to fetch datasets again.
3. Restart the kernel and run the updated notebook from the first cell. Its patched default `ROOT` is `/kaggle/working/jev_benchmark_v3_1`; do not reuse the previous manifest. If `/kaggle/working/jev_benchmark_v3` is still present, its data snapshots are reused automatically. Old model scores are not copied. Preserve/download files before any Kaggle session reset that clears working storage.
4. Run ML, then the separate paid Jev cell. API calls are not made while testing ML. The test/policy caps and cost controls are unchanged. Eight threads do not override the request-start spacing.

The new models use the same test selection as original v2. If you already inspected those test scores, disclose this as a computational revision on that holdout, not a new untouched test. This speed revision was driven by runtime, not selecting models on test scores.

There is no verified 30-minute guarantee on Kaggle. The structural workload is reduced substantially, but full runtime depends on CPUs, T4 availability, class counts and convergence. Actual two-T4 testing must happen in Kaggle. Pin and report the fast protocol with results; do not describe its search as best-possible ML optimization.


In [4]:
OVERVIEW = prepare_suite(ROOT, CFG, pilot_root=PILOT_ROOT)
display(OVERVIEW)


Prepare datasets:   0%|          | 0/8 [00:00<?, ?it/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Jev upper bound before cache/retries: 52,410 requests; retry ceiling: 157,230.
The cost guard is an input-token estimate, not a billing cap. Verify current API pricing before running Jev.


,dataset,seed,classes,train,validation,policy,test,test_class_counts,excluded_pilot_test_rows
0,AG News,2027,4,8000,1000,500,1000,"{'0': 250, '1': 250, '2': 250, '3': 250}",862
1,AG News,2028,4,8000,1000,500,1000,"{'0': 250, '1': 250, '2': 250, '3': 250}",862
2,AG News,2029,4,8000,1000,500,1000,"{'0': 250, '1': 250, '2': 250, '3': 250}",862
3,Banking77,2027,77,6001,1000,500,1500,"{'0': 19, '1': 19, '2': 20, '3': 21, '4': 20, ...",819
4,Banking77,2028,77,6001,1000,500,1500,"{'0': 19, '1': 19, '2': 20, '3': 21, '4': 20, ...",819
5,Banking77,2029,77,6001,1000,500,1500,"{'0': 19, '1': 19, '2': 20, '3': 21, '4': 20, ...",819
6,SMS Spam,2027,2,2502,834,500,1000,"{'0': 874, '1': 126}",842
7,SMS Spam,2028,2,2502,834,500,1000,"{'0': 874, '1': 126}",842
8,SMS Spam,2029,2,2502,834,500,1000,"{'0': 874, '1': 126}",842
9,IMDb,2027,2,8000,1000,500,1000,"{'0': 498, '1': 502}",891


## Fit and tune conventional models

GPU probes precede training. Successful per-model checkpoints allow rerunning this cell after interruption. Individual trial warnings/errors are saved under `runs/`; inspect them before publishing. Most scikit-learn models use CPUs. Larger text forests can take hours.

In [5]:
ML_STATUS = run_ml(ROOT, CFG)
display(ML_STATUS)
TABLES = render_results(ROOT, CFG)
display(TABLES['raw_balanced_accuracy'])
display(TABLES['adjusted_balanced_accuracy'])

Isolated GPU assignments: {0: 'GPU-6022ebf1-d6d9-94ff-266d-79b4f7c6b42b', 1: 'GPU-299dc466-f487-0454-c6a5-11edde7c7edd'}
[GPU lane 0 / probe]
Tesla T4
Tesla T4
GPU 0: XGBoost and CatBoost probe passed
[GPU lane 1 / probe]
Tesla T4
Tesla T4
GPU 0: XGBoost and CatBoost probe passed
[GPU lane 0 / probe]
cuML probe: local GPU 0, Logistic regression, text=False, weighted=False
[GPU lane 1 / probe]
cuML probe: local GPU 0, Logistic regression, text=False, weighted=False
[GPU lane 0 / probe]
cuML probe: local GPU 0, Logistic regression, text=False, weighted=True
cuML probe: local GPU 0, Logistic regression, text=True, weighted=False
[GPU lane 1 / probe]
cuML probe: local GPU 0, Logistic regression, text=False, weighted=True
cuML probe: local GPU 0, Logistic regression, text=True, weighted=False
[GPU lane 0 / probe]
cuML probe: local GPU 0, Logistic regression, text=True, weighted=True
cuML probe: local GPU 0, Random forest, text=True, weighted=False
/usr/local/lib/python3.12/dist-packages/cum

[{'dataset': 'AG News', 'seed': 2027, 'status': 'complete'},
 {'dataset': 'AG News', 'seed': 2029, 'status': 'complete'},
 {'dataset': 'Banking77', 'seed': 2028, 'status': 'complete'},
 {'dataset': 'SMS Spam', 'seed': 2027, 'status': 'complete'},
 {'dataset': 'SMS Spam', 'seed': 2029, 'status': 'complete'},
 {'dataset': 'IMDb', 'seed': 2028, 'status': 'complete'},
 {'dataset': 'Bank Marketing', 'seed': 2027, 'status': 'complete'},
 {'dataset': 'Bank Marketing', 'seed': 2029, 'status': 'complete'},
 {'dataset': 'Online Shoppers', 'seed': 2028, 'status': 'complete'},
 {'dataset': 'Breast Cancer', 'seed': 2027, 'status': 'complete'},
 {'dataset': 'Breast Cancer', 'seed': 2029, 'status': 'complete'},
 {'dataset': 'Iris', 'seed': 2028, 'status': 'complete'},
 {'dataset': 'AG News', 'seed': 2028, 'status': 'complete'},
 {'dataset': 'Banking77', 'seed': 2027, 'status': 'complete'},
 {'dataset': 'Banking77', 'seed': 2029, 'status': 'complete'},
 {'dataset': 'SMS Spam', 'seed': 2028, 'status': 

,Logistic regression,SVM,Decision tree,Random forest,Extra trees,k-NN,Naive Bayes,Hist gradient boost,XGBoost,CatBoost,Voting ensemble,Majority baseline,Jev zero-shot,Jev few-shot
dataset,,,,,,,,,,,,,,
AG News,87.4 ± 0.8 (n=3),88.4 ± 0.3 (n=3),67.5 ± 0.9 (n=3),71.6 ± 1.1 (n=3),75.1 ± 0.7 (n=3),78.4 ± 0.3 (n=3),87.4 ± 0.2 (n=3),80.3 ± 0.8 (n=3),82.6 ± 0.2 (n=3),82.0 ± 1.0 (n=3),87.0 ± 1.0 (n=3),25.0 ± 0.0 (n=3),pending,pending
Banking77,89.4 ± 0.2 (n=3),89.7 ± 0.6 (n=3),62.2 ± 0.9 (n=3),68.2 ± 1.4 (n=3),69.8 ± 0.4 (n=3),58.4 ± 1.9 (n=3),86.0 ± 0.5 (n=3),62.2 ± 1.2 (n=3),71.9 ± 0.4 (n=3),68.1 ± 0.7 (n=3),86.2 ± 0.4 (n=3),1.3 ± 0.0 (n=3),pending,pending
SMS Spam,86.4 ± 8.1 (n=3),93.7 ± 0.0 (n=3),85.7 ± 2.8 (n=3),89.0 ± 0.3 (n=3),88.7 ± 0.8 (n=3),87.8 ± 2.8 (n=3),95.0 ± 1.9 (n=3),92.1 ± 0.1 (n=3),88.3 ± 2.8 (n=3),90.7 ± 1.6 (n=3),89.0 ± 0.8 (n=3),50.0 ± 0.0 (n=3),pending,pending
IMDb,88.4 ± 0.2 (n=3),87.8 ± 0.9 (n=3),70.7 ± 1.1 (n=3),81.2 ± 0.8 (n=3),83.3 ± 1.1 (n=3),79.9 ± 1.5 (n=3),86.2 ± 0.4 (n=3),83.2 ± 0.3 (n=3),83.6 ± 0.2 (n=3),84.0 ± 0.2 (n=3),87.3 ± 0.2 (n=3),50.0 ± 0.0 (n=3),pending,pending
Bank Marketing,58.0 ± 0.6 (n=3),71.8 ± 0.7 (n=3),66.0 ± 4.8 (n=3),59.3 ± 1.4 (n=3),60.2 ± 0.3 (n=3),55.5 ± 0.5 (n=3),71.0 ± 0.4 (n=3),63.9 ± 7.4 (n=3),64.1 ± 7.0 (n=3),62.3 ± 7.8 (n=3),61.4 ± 5.5 (n=3),50.0 ± 0.0 (n=3),pending,pending
Online Shoppers,63.8 ± 11.0 (n=3),69.1 ± 0.3 (n=3),60.3 ± 5.1 (n=3),56.8 ± 6.5 (n=3),53.3 ± 3.8 (n=3),51.2 ± 1.3 (n=3),59.5 ± 0.4 (n=3),51.8 ± 0.8 (n=3),65.1 ± 9.5 (n=3),63.7 ± 10.0 (n=3),63.5 ± 10.0 (n=3),50.0 ± 0.0 (n=3),pending,pending
Breast Cancer,99.5 ± 0.4 (n=3),100.0 ± 0.0 (n=3),94.0 ± 2.2 (n=3),96.6 ± 1.9 (n=3),97.4 ± 1.2 (n=3),100.0 ± 0.0 (n=3),92.2 ± 0.7 (n=3),97.3 ± 0.6 (n=3),97.7 ± 0.4 (n=3),98.0 ± 0.6 (n=3),99.1 ± 0.4 (n=3),50.0 ± 0.0 (n=3),pending,pending
Iris,100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),97.5 ± 2.1 (n=3),98.9 ± 1.9 (n=3),100.0 ± 0.0 (n=3),97.8 ± 3.9 (n=3),100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),95.7 ± 2.1 (n=3),97.5 ± 2.1 (n=3),100.0 ± 0.0 (n=3),33.3 ± 0.0 (n=3),pending,pending


,Logistic regression,SVM,Decision tree,Random forest,Extra trees,k-NN,Naive Bayes,Hist gradient boost,XGBoost,CatBoost,Voting ensemble,Majority baseline,Jev zero-shot,Jev few-shot
dataset,,,,,,,,,,,,,,
AG News,87.4 ± 0.8 (n=3),88.4 ± 0.3 (n=3),67.5 ± 0.9 (n=3),71.6 ± 1.1 (n=3),75.1 ± 0.7 (n=3),78.4 ± 0.3 (n=3),87.4 ± 0.2 (n=3),80.3 ± 0.8 (n=3),82.6 ± 0.2 (n=3),82.0 ± 1.0 (n=3),87.0 ± 1.0 (n=3),25.0 ± 0.0 (n=3),pending,pending
Banking77,89.4 ± 0.2 (n=3),89.7 ± 0.6 (n=3),62.2 ± 0.9 (n=3),68.2 ± 1.4 (n=3),69.8 ± 0.4 (n=3),58.4 ± 1.9 (n=3),86.0 ± 0.5 (n=3),62.2 ± 1.2 (n=3),71.9 ± 0.4 (n=3),68.1 ± 0.7 (n=3),86.2 ± 0.4 (n=3),1.3 ± 0.0 (n=3),pending,pending
SMS Spam,95.6 ± 0.7 (n=3),96.2 ± 0.8 (n=3),88.5 ± 0.6 (n=3),96.2 ± 0.9 (n=3),94.5 ± 0.8 (n=3),92.4 ± 0.8 (n=3),96.3 ± 0.6 (n=3),93.8 ± 0.3 (n=3),90.7 ± 1.2 (n=3),94.1 ± 0.6 (n=3),95.3 ± 1.1 (n=3),50.0 ± 0.0 (n=3),pending,pending
IMDb,88.3 ± 0.2 (n=3),87.0 ± 1.9 (n=3),70.5 ± 0.6 (n=3),81.3 ± 0.7 (n=3),83.0 ± 1.5 (n=3),80.6 ± 1.5 (n=3),85.3 ± 0.9 (n=3),83.1 ± 0.2 (n=3),83.7 ± 0.2 (n=3),83.6 ± 0.5 (n=3),86.8 ± 0.3 (n=3),50.0 ± 0.0 (n=3),pending,pending
Bank Marketing,68.9 ± 2.4 (n=3),71.4 ± 2.0 (n=3),68.2 ± 2.5 (n=3),73.0 ± 1.2 (n=3),72.1 ± 0.1 (n=3),66.9 ± 0.4 (n=3),64.9 ± 2.6 (n=3),72.4 ± 1.2 (n=3),71.4 ± 2.5 (n=3),70.1 ± 2.8 (n=3),73.3 ± 0.3 (n=3),50.0 ± 0.0 (n=3),pending,pending
Online Shoppers,69.3 ± 1.1 (n=3),68.4 ± 0.4 (n=3),63.6 ± 0.2 (n=3),69.7 ± 0.6 (n=3),68.4 ± 1.2 (n=3),65.9 ± 1.5 (n=3),64.5 ± 2.3 (n=3),69.2 ± 3.5 (n=3),69.5 ± 1.4 (n=3),69.8 ± 1.9 (n=3),71.2 ± 1.4 (n=3),50.0 ± 0.0 (n=3),pending,pending
Breast Cancer,100.0 ± 0.0 (n=3),99.6 ± 0.7 (n=3),93.5 ± 1.4 (n=3),96.9 ± 1.5 (n=3),97.1 ± 0.5 (n=3),99.5 ± 0.8 (n=3),93.3 ± 1.6 (n=3),96.0 ± 2.5 (n=3),98.0 ± 0.5 (n=3),97.9 ± 0.7 (n=3),98.7 ± 0.7 (n=3),50.0 ± 0.0 (n=3),pending,pending
Iris,100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),97.5 ± 2.1 (n=3),98.9 ± 1.9 (n=3),100.0 ± 0.0 (n=3),97.8 ± 3.9 (n=3),100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),95.7 ± 2.1 (n=3),97.5 ± 2.1 (n=3),100.0 ± 0.0 (n=3),33.3 ± 0.0 (n=3),pending,pending


## Run Jev — paid API calls

The secret is read from Kaggle Secrets. tqdm shows each test/policy partition; exact successful requests are cached. Permanent API errors stop execution; exhausted transient retries count as failed predictions. Model IDs, request attempts and cache hits are recorded.

Binary **adjusted Jev uses labeled policy data to select a threshold**. Only the raw zero-shot panel is an end-to-end zero-shot baseline. Few-shot examples are sampled from training, one per class. Budget estimates exclude output-token charges; verify current pricing and account limits before executing this cell.

In [6]:
API_STATUS = run_jev(ROOT, CFG)
display(API_STATUS)

AG News 2027 zero-shot: test:   0%|          | 0/1000 [00:00<?, ?it/s]

AG News 2027 few-shot: test:   0%|          | 0/1000 [00:00<?, ?it/s]

AG News 2028 zero-shot: test:   0%|          | 0/1000 [00:00<?, ?it/s]

AG News 2028 few-shot: test:   0%|          | 0/1000 [00:00<?, ?it/s]

AG News 2029 zero-shot: test:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Iris 2028 zero-shot: test:   0%|          | 0/30 [00:00<?, ?it/s]

Iris 2028 few-shot: test:   0%|          | 0/30 [00:00<?, ?it/s]

Iris 2029 zero-shot: test:   0%|          | 0/30 [00:00<?, ?it/s]

Iris 2029 few-shot: test:   0%|          | 0/30 [00:00<?, ?it/s]

{'attempts': 38922, 'estimated_input_cost': 4.1905409279999235}

## Compare and export

Rows are datasets; columns are models. Values are mean ± sample SD across training seeds, **not confidence intervals**. Both panels use the same test cases. The report includes accuracy/macro-F1 and paired bootstrap differences; JSON records include minority recall, confusion matrices, thresholds and raw probabilities. Publish both raw and adjusted panels with the protocol.

Save/download the output ZIP to preserve caches and resume later. It contains benchmark artifacts and dataset snapshots, never the API key.

In [7]:
TABLES = render_results(ROOT, CFG)
display(Markdown('### Raw decision rules'))
display(TABLES['raw_balanced_accuracy'])
display(Markdown('### Binary thresholds selected on policy data'))
display(TABLES['adjusted_balanced_accuracy'])
display(FileLink(str(ROOT / 'report.html')))
import shutil
# Include the source/protocol used in the result download for reproducibility.
shutil.copytree(CODE_DIR / 'jevbench', ROOT / 'source' / 'jevbench', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__'))
for filename in ['BENCHMARK_V2.md', 'BENCHMARK_V3.md', 'requirements-v2.txt']:
    shutil.copy2(CODE_DIR / filename, ROOT / filename)
archive = shutil.make_archive(str(ROOT) + '_results', 'zip', root_dir=ROOT.parent, base_dir=ROOT.name)
display(FileLink(archive))

### Raw decision rules

,Logistic regression,SVM,Decision tree,Random forest,Extra trees,k-NN,Naive Bayes,Hist gradient boost,XGBoost,CatBoost,Voting ensemble,Majority baseline,Jev zero-shot,Jev few-shot
dataset,,,,,,,,,,,,,,
AG News,87.4 ± 0.8 (n=3),88.4 ± 0.3 (n=3),67.5 ± 0.9 (n=3),71.6 ± 1.1 (n=3),75.1 ± 0.7 (n=3),78.4 ± 0.3 (n=3),87.4 ± 0.2 (n=3),80.3 ± 0.8 (n=3),82.6 ± 0.2 (n=3),82.0 ± 1.0 (n=3),87.0 ± 1.0 (n=3),25.0 ± 0.0 (n=3),87.5 ± 0.0 (n=3),86.3 ± 0.6 (n=3)
Banking77,89.4 ± 0.2 (n=3),89.7 ± 0.6 (n=3),62.2 ± 0.9 (n=3),68.2 ± 1.4 (n=3),69.8 ± 0.4 (n=3),58.4 ± 1.9 (n=3),86.0 ± 0.5 (n=3),62.2 ± 1.2 (n=3),71.9 ± 0.4 (n=3),68.1 ± 0.7 (n=3),86.2 ± 0.4 (n=3),1.3 ± 0.0 (n=3),78.9 ± 0.0 (n=3),81.9 ± 1.7 (n=3)
SMS Spam,86.4 ± 8.1 (n=3),93.7 ± 0.0 (n=3),85.7 ± 2.8 (n=3),89.0 ± 0.3 (n=3),88.7 ± 0.8 (n=3),87.8 ± 2.8 (n=3),95.0 ± 1.9 (n=3),92.1 ± 0.1 (n=3),88.3 ± 2.8 (n=3),90.7 ± 1.6 (n=3),89.0 ± 0.8 (n=3),50.0 ± 0.0 (n=3),96.1 ± 0.0 (n=3),95.6 ± 0.9 (n=3)
IMDb,88.4 ± 0.2 (n=3),87.8 ± 0.9 (n=3),70.7 ± 1.1 (n=3),81.2 ± 0.8 (n=3),83.3 ± 1.1 (n=3),79.9 ± 1.5 (n=3),86.2 ± 0.4 (n=3),83.2 ± 0.3 (n=3),83.6 ± 0.2 (n=3),84.0 ± 0.2 (n=3),87.3 ± 0.2 (n=3),50.0 ± 0.0 (n=3),96.3 ± 0.0 (n=3),95.9 ± 0.5 (n=3)
Bank Marketing,58.0 ± 0.6 (n=3),71.8 ± 0.7 (n=3),66.0 ± 4.8 (n=3),59.3 ± 1.4 (n=3),60.2 ± 0.3 (n=3),55.5 ± 0.5 (n=3),71.0 ± 0.4 (n=3),63.9 ± 7.4 (n=3),64.1 ± 7.0 (n=3),62.3 ± 7.8 (n=3),61.4 ± 5.5 (n=3),50.0 ± 0.0 (n=3),53.4 ± 0.0 (n=3),55.3 ± 3.1 (n=3)
Online Shoppers,63.8 ± 11.0 (n=3),69.1 ± 0.3 (n=3),60.3 ± 5.1 (n=3),56.8 ± 6.5 (n=3),53.3 ± 3.8 (n=3),51.2 ± 1.3 (n=3),59.5 ± 0.4 (n=3),51.8 ± 0.8 (n=3),65.1 ± 9.5 (n=3),63.7 ± 10.0 (n=3),63.5 ± 10.0 (n=3),50.0 ± 0.0 (n=3),51.4 ± 0.0 (n=3),54.7 ± 9.5 (n=3)
Breast Cancer,99.5 ± 0.4 (n=3),100.0 ± 0.0 (n=3),94.0 ± 2.2 (n=3),96.6 ± 1.9 (n=3),97.4 ± 1.2 (n=3),100.0 ± 0.0 (n=3),92.2 ± 0.7 (n=3),97.3 ± 0.6 (n=3),97.7 ± 0.4 (n=3),98.0 ± 0.6 (n=3),99.1 ± 0.4 (n=3),50.0 ± 0.0 (n=3),61.0 ± 0.0 (n=3),88.8 ± 5.5 (n=3)
Iris,100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),97.5 ± 2.1 (n=3),98.9 ± 1.9 (n=3),100.0 ± 0.0 (n=3),97.8 ± 3.9 (n=3),100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),95.7 ± 2.1 (n=3),97.5 ± 2.1 (n=3),100.0 ± 0.0 (n=3),33.3 ± 0.0 (n=3),97.0 ± 0.0 (n=3),94.5 ± 4.8 (n=3)


### Binary thresholds selected on policy data

,Logistic regression,SVM,Decision tree,Random forest,Extra trees,k-NN,Naive Bayes,Hist gradient boost,XGBoost,CatBoost,Voting ensemble,Majority baseline,Jev zero-shot,Jev few-shot
dataset,,,,,,,,,,,,,,
AG News,87.4 ± 0.8 (n=3),88.4 ± 0.3 (n=3),67.5 ± 0.9 (n=3),71.6 ± 1.1 (n=3),75.1 ± 0.7 (n=3),78.4 ± 0.3 (n=3),87.4 ± 0.2 (n=3),80.3 ± 0.8 (n=3),82.6 ± 0.2 (n=3),82.0 ± 1.0 (n=3),87.0 ± 1.0 (n=3),25.0 ± 0.0 (n=3),87.5 ± 0.0 (n=3),86.3 ± 0.6 (n=3)
Banking77,89.4 ± 0.2 (n=3),89.7 ± 0.6 (n=3),62.2 ± 0.9 (n=3),68.2 ± 1.4 (n=3),69.8 ± 0.4 (n=3),58.4 ± 1.9 (n=3),86.0 ± 0.5 (n=3),62.2 ± 1.2 (n=3),71.9 ± 0.4 (n=3),68.1 ± 0.7 (n=3),86.2 ± 0.4 (n=3),1.3 ± 0.0 (n=3),78.9 ± 0.0 (n=3),81.9 ± 1.7 (n=3)
SMS Spam,95.6 ± 0.7 (n=3),96.2 ± 0.8 (n=3),88.5 ± 0.6 (n=3),96.2 ± 0.9 (n=3),94.5 ± 0.8 (n=3),92.4 ± 0.8 (n=3),96.3 ± 0.6 (n=3),93.8 ± 0.3 (n=3),90.7 ± 1.2 (n=3),94.1 ± 0.6 (n=3),95.3 ± 1.1 (n=3),50.0 ± 0.0 (n=3),95.9 ± 0.7 (n=3),95.8 ± 0.5 (n=3)
IMDb,88.3 ± 0.2 (n=3),87.0 ± 1.9 (n=3),70.5 ± 0.6 (n=3),81.3 ± 0.7 (n=3),83.0 ± 1.5 (n=3),80.6 ± 1.5 (n=3),85.3 ± 0.9 (n=3),83.1 ± 0.2 (n=3),83.7 ± 0.2 (n=3),83.6 ± 0.5 (n=3),86.8 ± 0.3 (n=3),50.0 ± 0.0 (n=3),96.1 ± 0.6 (n=3),95.5 ± 0.8 (n=3)
Bank Marketing,68.9 ± 2.4 (n=3),71.4 ± 2.0 (n=3),68.2 ± 2.5 (n=3),73.0 ± 1.2 (n=3),72.1 ± 0.1 (n=3),66.9 ± 0.4 (n=3),64.9 ± 2.6 (n=3),72.4 ± 1.2 (n=3),71.4 ± 2.5 (n=3),70.1 ± 2.8 (n=3),73.3 ± 0.3 (n=3),50.0 ± 0.0 (n=3),59.7 ± 1.6 (n=3),59.0 ± 2.5 (n=3)
Online Shoppers,69.3 ± 1.1 (n=3),68.4 ± 0.4 (n=3),63.6 ± 0.2 (n=3),69.7 ± 0.6 (n=3),68.4 ± 1.2 (n=3),65.9 ± 1.5 (n=3),64.5 ± 2.3 (n=3),69.2 ± 3.5 (n=3),69.5 ± 1.4 (n=3),69.8 ± 1.9 (n=3),71.2 ± 1.4 (n=3),50.0 ± 0.0 (n=3),51.8 ± 0.1 (n=3),53.6 ± 7.3 (n=3)
Breast Cancer,100.0 ± 0.0 (n=3),99.6 ± 0.7 (n=3),93.5 ± 1.4 (n=3),96.9 ± 1.5 (n=3),97.1 ± 0.5 (n=3),99.5 ± 0.8 (n=3),93.3 ± 1.6 (n=3),96.0 ± 2.5 (n=3),98.0 ± 0.5 (n=3),97.9 ± 0.7 (n=3),98.7 ± 0.7 (n=3),50.0 ± 0.0 (n=3),88.4 ± 2.2 (n=3),92.7 ± 0.9 (n=3)
Iris,100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),97.5 ± 2.1 (n=3),98.9 ± 1.9 (n=3),100.0 ± 0.0 (n=3),97.8 ± 3.9 (n=3),100.0 ± 0.0 (n=3),100.0 ± 0.0 (n=3),95.7 ± 2.1 (n=3),97.5 ± 2.1 (n=3),100.0 ± 0.0 (n=3),33.3 ± 0.0 (n=3),97.0 ± 0.0 (n=3),94.5 ± 4.8 (n=3)


/kaggle/working/jev_benchmark_v3_1/report.html

/kaggle/working/jev_benchmark_v3_1_results.zip